# Day 2: Data Cleaning, Missing Value Treatment and Outlier Handling

## Objective

The objective of this stage is to improve data quality by fixing data types, handling missing values, removing duplicate records, detecting outliers, and preparing a clean dataset for further exploratory analysis.

## Tasks To Performe

1. Fix data types
2. Standardize categorical variables
3. Remove duplicate records
4. Handle missing values
5. Detect outliers using IQR method
6. Apply transformations if required
7. Save cleaned dataset

In [250]:
import pandas as pd
import numpy as np

### Purpose

Import the required libraries for data cleaning and preprocessing.

### Key Findings

- Pandas provides functions for data manipulation and preprocessing.
- NumPy supports numerical operations and calculations.

### Interpretation

These libraries form the foundation for cleaning, transforming, and preparing the dataset for analysis.

### Business Insight

Reliable preprocessing is essential because business decisions are only as accurate as the underlying data.

### Conclusion

The required libraries have been successfully imported and the environment is ready for preprocessing.

In [251]:
df = pd.read_csv("../data/raw/hotel_bookings.csv")



In [252]:
df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


### Purpose

The `head()` function is used to verify that the dataset has been loaded correctly and to obtain an initial view of the records before preprocessing begins.

### Key Findings

- The dataset contains hotel booking information, customer details, stay duration, pricing information, and reservation outcomes.
- Both numerical and categorical features are present.
- Missing values can already be observed in columns such as `agent` and `company`.
- Date-related information is available through multiple columns.

### Interpretation

The dataset structure appears consistent and suitable for preprocessing. The presence of different feature types indicates that multiple cleaning techniques may be required.

### Business Insight

The available features provide sufficient information to analyze booking behavior, customer preferences, hotel demand, pricing patterns, and cancellation trends.

### Conclusion

The dataset has been loaded successfully and is ready for data cleaning and preprocessing.

In [253]:
df.shape

(119390, 32)

### Purpose

The `shape` attribute is used to determine the number of rows and columns present in the dataset before preprocessing.

### Key Findings

- The dataset contains 119,390 records and 32 features.
- The dataset size is sufficiently large for meaningful analysis.

### Interpretation

The row count represents individual hotel booking records, while the columns represent different booking, customer, pricing, and reservation attributes.

### Business Insight

A large dataset improves the reliability of trends and patterns identified during analysis, helping generate more dependable business insights.

### Conclusion

The dataset dimensions have been recorded and will be used as a reference to measure the impact of cleaning operations such as duplicate removal and missing value treatment.

In [254]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  object 
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  object 
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal            

### Purpose

The `info()` function is used to examine the dataset structure, identify data types, detect missing values, and assess overall data quality before preprocessing.

### Key Findings

- The dataset contains **119,390 booking records** and **32 features**.
- The dataset consists of:
  - 16 integer columns (`int64`)
  - 4 floating-point columns (`float64`)
  - 12 categorical columns (`object`)
- Missing values are present in:
  - `children` (4 missing values)
  - `country` (488 missing values)
  - `agent` (16,340 missing values)
  - `company` (112,593 missing values)


### Interpretation

The dataset is well-structured and contains a balanced combination of numerical and categorical features. Most columns are complete, with missing values concentrated in only a few variables. Additionally, some columns require datatype correction before further analysis.

### Business Insight

The large number of missing values in the `company` column suggests that most bookings were not associated with corporate clients. Similarly, missing values in the `agent` column may indicate direct bookings made without travel agents. Understanding these patterns can provide insights into booking channels and customer behavior.

### Conclusion

The dataset is suitable for preprocessing and analysis. Missing values have been identified, and datatype corrections will be performed before proceeding with further cleaning steps.

In [255]:
df['reservation_status_date'] = pd.to_datetime(df['reservation_status_date'])

In [256]:
df[['reservation_status_date']].dtypes

reservation_status_date    datetime64[ns]
dtype: object

### Purpose

Convert columns to their appropriate data types to improve data consistency and support accurate analysis.

### Key Findings

- The `reservation_status_date` column was originally stored as an object data type.
- The column was successfully converted to `datetime64[ns]`.

### Interpretation

Date values stored as text limit date calculations and time-based analysis. Converting them to datetime format allows efficient handling of temporal information.

### Business Insight

Correct date formats enable the analysis of reservation trends, seasonal demand patterns, booking behavior, and hotel performance over time.

### Conclusion

The `reservation_status_date` column was successfully converted to datetime format, improving the dataset's readiness for preprocessing and future analysis.

In [257]:
categorical_cols = df.select_dtypes(include='object').columns

categorical_cols

Index(['hotel', 'arrival_date_month', 'meal', 'country', 'market_segment',
       'distribution_channel', 'reserved_room_type', 'assigned_room_type',
       'deposit_type', 'customer_type', 'reservation_status'],
      dtype='object')

### Purpose

Identify all categorical features available in the dataset before performing standardization and cleaning.

### Key Findings

- The dataset contains 11 categorical variables.
- These features describe hotel type, booking channels, customer categories, room allocation, reservation outcomes, and geographical information.
- Examples include `hotel`, `country`, `market_segment`, `customer_type`, and `reservation_status`.

### Interpretation

Categorical variables represent groups and classifications rather than numerical measurements. These features play an important role in customer segmentation and behavioral analysis.

### Business Insight

Understanding categorical features helps analyze customer preferences, booking sources, hotel performance, reservation outcomes, and market trends.

### Conclusion

Categorical features were successfully identified and will be reviewed for consistency before further analysis.

In [258]:
for col in categorical_cols:
    print(f"\n{col}")
    print(df[col].unique()[:10])


hotel
['Resort Hotel' 'City Hotel']

arrival_date_month
['July' 'August' 'September' 'October' 'November' 'December' 'January'
 'February' 'March' 'April']

meal
['BB' 'FB' 'HB' 'SC' 'Undefined']

country
['PRT' 'GBR' 'USA' 'ESP' 'IRL' 'FRA' nan 'ROU' 'NOR' 'OMN']

market_segment
['Direct' 'Corporate' 'Online TA' 'Offline TA/TO' 'Complementary' 'Groups'
 'Undefined' 'Aviation']

distribution_channel
['Direct' 'Corporate' 'TA/TO' 'Undefined' 'GDS']

reserved_room_type
['C' 'A' 'D' 'E' 'G' 'F' 'H' 'L' 'P' 'B']

assigned_room_type
['C' 'A' 'D' 'E' 'G' 'F' 'I' 'B' 'H' 'P']

deposit_type
['No Deposit' 'Refundable' 'Non Refund']

customer_type
['Transient' 'Contract' 'Transient-Party' 'Group']

reservation_status
['Check-Out' 'Canceled' 'No-Show']


### Purpose

The unique values of categorical variables were examined to understand the available categories, identify inconsistencies, detect missing or undefined values, and determine whether standardization was required.

### Key Findings

- The dataset contains several categorical variables related to hotel operations, customer information, booking channels, and reservation outcomes.
- Most categorical variables contained meaningful and consistent categories.
- The value **"Undefined"** was identified in:
  - `meal`
  - `market_segment`
  - `distribution_channel`
- Missing values (`NaN`) were observed in the `country` column.
- Variables such as `hotel`, `customer_type`, `deposit_type`, and `reservation_status` contained clearly defined categories suitable for analysis.

### Interpretation

Reviewing unique values helps identify data quality issues before analysis. The presence of categories such as **"Undefined"** indicates that some values may require standardization to improve consistency and interpretability.

### Business Insight

Categorical variables provide important information about customer behavior, booking sources, hotel types, room allocation, and reservation outcomes. Ensuring category consistency improves segmentation analysis and supports more reliable business insights.

### Conclusion

The categorical variables were successfully reviewed. A small number of non-informative categories and missing values were identified and marked for cleaning and standardization in subsequent preprocessing steps.

In [259]:
df['meal'] = df['meal'].replace('Undefined', 'SC')

df['market_segment'] = df['market_segment'].replace('Undefined', 'Online TA')

df['distribution_channel'] = df['distribution_channel'].replace('Undefined', 'TA/TO')

In [260]:
df['meal'].unique()

df['market_segment'].unique()

df['distribution_channel'].unique()

array(['Direct', 'Corporate', 'TA/TO', 'GDS'], dtype=object)

### Purpose

Standardization is the process of making categorical values consistent throughout the dataset. It helps eliminate ambiguous, inconsistent, or non-informative categories that may affect analysis.

### Key Findings

- Most categorical variables were already consistent.
- The value `Undefined` was found in:
  - `meal`
  - `market_segment`
  - `distribution_channel`
- These values were standardized by replacing them with meaningful categories.

### Interpretation

The category `Undefined` does not provide useful business information and can create separate groups during analysis. Standardizing such values improves data consistency and makes the categories easier to interpret.

### Business Insight

Consistent categories improve customer segmentation, booking channel analysis, and service preference analysis. This leads to more accurate business insights and reporting.

### Conclusion

Categorical variables were reviewed and standardized to ensure consistency, improve interpretability, and support reliable analysis.

In [261]:
df.duplicated().sum()

np.int64(31994)

### Use Case

The `duplicated()` function is used to identify records that appear more than once in the dataset. Duplicate records can affect statistical analysis, visualizations, and business conclusions.

### Key Findings

- A total of **31,994 duplicate records** were identified in the dataset.
- This represents approximately **26.8% of all booking records**.

### Interpretation

The presence of a large number of duplicate rows indicates that several booking records contain identical information across all columns. If these duplicates are retained, they can artificially increase booking counts and distort analytical results.

### Business Insight

Accurate hotel analysis requires each row to represent a unique booking transaction. Duplicate records may lead to overestimation of customer demand, occupancy levels, booking volume, and revenue-related metrics.

### Conclusion

A significant number of duplicate records were identified. These records should be removed to ensure data quality and improve the reliability of subsequent analysis.

In [262]:
df = df.drop_duplicates()

In [263]:
df.duplicated().sum()

np.int64(0)

In [264]:
df.shape

(87396, 32)

### Use Case

The `drop_duplicates()` function is used to remove duplicate records from the dataset. This ensures that each row represents a unique booking transaction and prevents duplicate observations from influencing analysis.

### Key Findings

- A total of **31,994 duplicate records** were removed from the dataset.
- The dataset size reduced from **119,390 rows** to **87,396 rows**.
- After removal, the dataset contains **0 duplicate records**.

### Interpretation

The large number of duplicate records indicates that many bookings contained identical information across all features. Keeping these records would have resulted in repeated observations and biased statistical analysis.

### Business Insight

Hotel performance metrics such as booking volume, customer demand, occupancy estimation, and cancellation analysis should be based on unique transactions. Removing duplicates improves the accuracy and reliability of business insights.

### Conclusion

Duplicate records were successfully removed, resulting in a cleaner dataset with 87,396 unique booking records ready for further preprocessing and analysis.

In [265]:
missing_values = df.isnull().sum()

missing_values[missing_values > 0]

children        4
country       452
agent       12193
company     82137
dtype: int64

### Use Case

The `isnull().sum()` function is used to identify missing values in the dataset. Missing values can reduce data quality and affect the accuracy of statistical analysis and machine learning models.

### Key Findings

The following columns contain missing values:

- `children` → 4 missing values
- `country` → 452 missing values
- `agent` → 12,193 missing values
- `company` → 82,137 missing values

### Interpretation

The dataset contains missing values in both numerical and categorical features. The `company` column has the highest number of missing values, while `children` contains only a few missing observations.

### Business Insight

The large number of missing values in the `company` column suggests that most bookings were not associated with a company booking source. Similarly, missing values in the `agent` column may indicate bookings made directly without travel agents. Understanding these patterns provides insights into customer booking channels and booking behavior.

### Conclusion

Missing values were identified successfully and appropriate treatment strategies will be applied based on the nature and business meaning of each column.

In [266]:
(df.isnull().sum() / len(df)) * 100

hotel                              0.000000
is_canceled                        0.000000
lead_time                          0.000000
arrival_date_year                  0.000000
arrival_date_month                 0.000000
arrival_date_week_number           0.000000
arrival_date_day_of_month          0.000000
stays_in_weekend_nights            0.000000
stays_in_week_nights               0.000000
adults                             0.000000
children                           0.004577
babies                             0.000000
meal                               0.000000
country                            0.517186
market_segment                     0.000000
distribution_channel               0.000000
is_repeated_guest                  0.000000
previous_cancellations             0.000000
previous_bookings_not_canceled     0.000000
reserved_room_type                 0.000000
assigned_room_type                 0.000000
booking_changes                    0.000000
deposit_type                    

In [267]:
missing_percentage = (df.isnull().sum() / len(df)) * 100

missing_percentage[missing_percentage > 0]

children     0.004577
country      0.517186
agent       13.951439
company     93.982562
dtype: float64

### Use Case

Calculating missing value percentages helps evaluate the severity of missing data and supports the selection of appropriate treatment methods such as imputation, replacement, or column removal.

### Key Findings

The percentage of missing values in the dataset is:

- `children` → 0.005%
- `country` → 0.52%
- `agent` → 13.95%
- `company` → 93.98%

### Interpretation

- `children` contains a negligible number of missing values and can be safely imputed.
- `country` has less than 1% missing values and can be filled using the most frequent category.
- `agent` contains a moderate level of missing values and requires appropriate treatment.
- `company` contains missing values in nearly 94% of records, indicating that the column has very limited usable information.

### Business Insight

The extremely high missing percentage in the `company` column suggests that most hotel bookings were not associated with corporate bookings. Similarly, missing values in the `agent` column may indicate direct bookings without travel agents.

### Conclusion

Different missing value treatment strategies will be applied based on the percentage and business meaning of each column to preserve data quality and analytical reliability.

In [268]:
df.drop('company', axis=1, inplace=True)

In [269]:
df['children'].mode()

0    0.0
Name: children, dtype: float64

In [270]:
df['country'].mode()

0    PRT
Name: country, dtype: object

In [271]:
df['agent'].describe()

count    75203.000000
mean        94.138306
std        113.188172
min          1.000000
25%          9.000000
50%         14.000000
75%        240.000000
max        535.000000
Name: agent, dtype: float64

### Use Case

Before imputing missing values, the distribution and characteristics of affected columns were examined to select appropriate treatment methods. The goal is to preserve data quality while minimizing the impact of missing information.

### Key Findings

- `company` contained 93.98% missing values and was removed from the dataset.
- The mode of `children` is **0**, indicating that most bookings did not include children.
- The mode of `country` is **PRT (Portugal)**, making it the most frequent country in the dataset.
- The `agent` column contains a moderate number of missing values and shows a wide range of values, with a median of **14**.

### Interpretation

- The `company` column contains insufficient information due to the extremely high percentage of missing values and therefore provides limited analytical value.
- Missing values in `children` can reasonably be replaced with 0 because most bookings do not include children.
- Missing values in `country` can be replaced using the mode (PRT), which represents the most common booking origin.
- Since the `agent` column contains numerical identifiers and exhibits variability, the median is a suitable imputation method because it is less sensitive to extreme values.

### Business Insight

The missing value patterns suggest that most bookings are individual customer bookings rather than company-sponsored reservations. The dominance of Portugal (PRT) indicates a strong domestic customer base. Proper treatment of missing values ensures accurate customer segmentation and booking behavior analysis.

### Conclusion

Missing value treatment decisions were selected based on data distribution, business meaning, and the percentage of missing information to ensure reliable downstream analysis.

In [272]:
df['children'] = df['children'].fillna(df['children'].mode()[0])

df['country'] = df['country'].fillna(df['country'].mode()[0])

df['agent'] = df['agent'].fillna(df['agent'].median())

In [273]:
df.isnull().sum()

hotel                             0
is_canceled                       0
lead_time                         0
arrival_date_year                 0
arrival_date_month                0
arrival_date_week_number          0
arrival_date_day_of_month         0
stays_in_weekend_nights           0
stays_in_week_nights              0
adults                            0
children                          0
babies                            0
meal                              0
country                           0
market_segment                    0
distribution_channel              0
is_repeated_guest                 0
previous_cancellations            0
previous_bookings_not_canceled    0
reserved_room_type                0
assigned_room_type                0
booking_changes                   0
deposit_type                      0
agent                             0
days_in_waiting_list              0
customer_type                     0
adr                               0
required_car_parking_spaces 

### Use Case

The `isnull().sum()` function was executed after missing value treatment to verify whether all missing values had been successfully handled.

### Key Findings

- All columns contain **0 missing values**.
- No null values remain in the dataset.
- The applied imputation and column removal strategies successfully resolved all missing data issues.

### Interpretation

The missing value treatment process was effective in eliminating incomplete records while preserving the integrity of the dataset. The dataset is now complete and suitable for further preprocessing and analysis.

### Business Insight

A dataset without missing values improves the reliability of customer behavior analysis, booking trend analysis, cancellation analysis, and revenue-related insights. Complete data reduces the risk of biased conclusions and improves analytical accuracy.

### Conclusion

All missing values were successfully treated and verified. The dataset is now clean, complete, and ready for outlier detection and further exploratory data analysis.

In [274]:
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

numerical_cols

['is_canceled',
 'lead_time',
 'arrival_date_year',
 'arrival_date_week_number',
 'arrival_date_day_of_month',
 'stays_in_weekend_nights',
 'stays_in_week_nights',
 'adults',
 'children',
 'babies',
 'is_repeated_guest',
 'previous_cancellations',
 'previous_bookings_not_canceled',
 'booking_changes',
 'agent',
 'days_in_waiting_list',
 'adr',
 'required_car_parking_spaces',
 'total_of_special_requests']

### Use Case

The `select_dtypes()` function was used to identify all numerical columns in the dataset. This step is important because statistical analysis, outlier detection, correlation analysis, and feature transformations are primarily performed on numerical variables.

### Key Findings

A total of 19 numerical columns were identified in the dataset, including variables related to:

- Booking behavior (`lead_time`, `booking_changes`)
- Stay duration (`stays_in_weekend_nights`, `stays_in_week_nights`)
- Guest information (`adults`, `children`, `babies`)
- Reservation history (`previous_cancellations`, `previous_bookings_not_canceled`)
- Pricing (`adr`)
- Waiting periods (`days_in_waiting_list`)

### Interpretation

The dataset contains a mixture of continuous variables, count-based variables, and binary indicators. While all of these columns are numerical in data type, not all of them are suitable for outlier detection. Variables such as `lead_time`, `adr`, and stay duration can naturally contain extreme values, whereas binary variables such as `is_canceled` and `is_repeated_guest` require different analytical treatment.

### Business Insight

Numerical variables capture important aspects of hotel operations, including customer booking patterns, room pricing, guest composition, and reservation behavior. Understanding these variables helps identify unusual customer behavior, pricing anomalies, and operational trends that can influence business decisions.

### Conclusion

The numerical features were successfully identified and prepared for further statistical analysis. The next step is to determine which of these variables are appropriate for outlier detection using the IQR method.

In [275]:
outlier_cols = [
    'lead_time',
    'stays_in_weekend_nights',
    'stays_in_week_nights',
    'adults',
    'children',
    'babies',
    'previous_cancellations',
    'previous_bookings_not_canceled',
    'booking_changes',
    'agent',
    'days_in_waiting_list',
    'adr'
]

outlier_cols

['lead_time',
 'stays_in_weekend_nights',
 'stays_in_week_nights',
 'adults',
 'children',
 'babies',
 'previous_cancellations',
 'previous_bookings_not_canceled',
 'booking_changes',
 'agent',
 'days_in_waiting_list',
 'adr']

### Use Case

Before applying the IQR method, relevant numerical variables were selected for outlier detection. This ensures that the analysis focuses only on variables where extreme values are meaningful and potentially impactful.

### Key Findings

The selected variables represent important business metrics such as:

- Booking lead time
- Stay duration
- Number of guests
- Booking modification history
- Waiting period before confirmation
- Room pricing (ADR)

### Interpretation

Not all numerical variables are suitable for outlier detection. Binary variables such as `is_canceled` and `is_repeated_guest`, as well as date-related variables such as `arrival_date_year`, were excluded because their values represent categories rather than continuous measurements.

The selected variables can legitimately contain unusually high or low values and therefore require outlier analysis.

### Business Insight

Extreme values in booking lead time, room price, stay duration, or waiting period may indicate unusual customer behavior, special booking situations, pricing anomalies, or operational exceptions. Identifying such observations helps improve data quality and analytical reliability.

### Conclusion

Relevant numerical variables were selected for outlier detection. These features will be analyzed using the IQR method to identify extreme observations that may influence statistical analysis and business insights.

In [276]:
for col in outlier_cols:
    
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    
    print(f"{col}: {len(outliers)} outliers")

lead_time: 2396 outliers
stays_in_weekend_nights: 220 outliers
stays_in_week_nights: 1531 outliers
adults: 22899 outliers
children: 8364 outliers
babies: 914 outliers
previous_cancellations: 1685 outliers
previous_bookings_not_canceled: 3545 outliers
booking_changes: 15902 outliers
agent: 0 outliers
days_in_waiting_list: 860 outliers
adr: 2490 outliers


## Use Case

The Interquartile Range (IQR) method was applied to identify potential outliers in selected numerical variables. Outliers are observations that lie significantly outside the typical range of the data and may influence statistical summaries, visualizations, and analytical outcomes.

## What is IQR?

The Interquartile Range (IQR) measures the spread of the middle 50% of the data.

- Q1 (25th Percentile): Value below which 25% of observations fall.
- Q3 (75th Percentile): Value below which 75% of observations fall.
- IQR = Q3 − Q1

Outlier Detection Rule:

- Lower Bound = Q1 − 1.5 × IQR
- Upper Bound = Q3 + 1.5 × IQR

Any value outside these bounds is considered a potential outlier.

## Key Findings

The IQR analysis detected outliers in multiple numerical variables:

| Variable | Outlier Count |
|----------|--------------:|
| adults | 22,899 |
| booking_changes | 15,902 |
| children | 8,364 |
| previous_bookings_not_canceled | 3,545 |
| adr | 2,490 |
| lead_time | 2,396 |
| previous_cancellations | 1,685 |
| stays_in_week_nights | 1,531 |
| babies | 914 |
| days_in_waiting_list | 860 |
| stays_in_weekend_nights | 220 |
| agent | 0 |

The highest number of outliers was observed in the `adults`, `booking_changes`, and `children` variables.

## Interpretation

The presence of outliers indicates that some hotel bookings differ substantially from typical booking behavior.

Examples include:

- Reservations involving unusually large groups of guests.
- Bookings with exceptionally high numbers of modifications.
- Customers with extensive booking histories.
- Extremely long lead times before arrival.
- Reservations with unusually high average daily rates (ADR).

The `agent` variable did not contain any outliers according to the IQR method, suggesting a relatively stable distribution.

It is important to note that outliers are not always data errors. In business datasets, they often represent rare but genuine events.

## Why Outlier Detection is Important

Outliers can:

- Distort measures such as mean and standard deviation.
- Affect data visualization and interpretation.
- Influence predictive model performance.
- Reveal unusual customer behavior and business opportunities.

Therefore, identifying outliers is an essential step before deciding whether treatment or transformation is required.

## Business Insight

The detected outliers provide valuable information about exceptional customer behavior.

Examples include:

- Guests making reservations far in advance (`lead_time`).
- Large family or group bookings (`adults`, `children`).
- Frequent booking modifications (`booking_changes`).
- Premium-priced reservations (`adr`).
- Loyal customers with extensive booking history (`previous_bookings_not_canceled`).

These observations may represent important customer segments and should be evaluated carefully before removal.

## Conclusion

The IQR method successfully identified potential outliers across several booking-related variables. Since many of these extreme values may represent legitimate business behavior rather than data quality issues, they were retained for further analysis. Additional investigation through skewness analysis was performed to determine whether data transformation was necessary.

In [277]:
df[outlier_cols].skew().sort_values(ascending=False)

previous_cancellations            34.323744
babies                            21.148519
previous_bookings_not_canceled    20.459725
adults                            19.859578
days_in_waiting_list              19.396419
adr                               10.921447
booking_changes                    5.546328
children                           3.463670
stays_in_week_nights               2.691321
lead_time                          1.431774
stays_in_weekend_nights            1.411637
agent                              1.121197
dtype: float64

## Use Case

Skewness analysis was performed to understand the distribution pattern of numerical variables and determine whether data transformation was required. Highly skewed variables can affect statistical analysis, visualization interpretation, and machine learning model performance.

## What is Skewness?

Skewness measures the asymmetry of a data distribution.

- Skewness = 0 → Perfectly symmetric (Normal Distribution)
- Skewness > 0 → Right-skewed (long tail on the right)
- Skewness < 0 → Left-skewed (long tail on the left)

General Interpretation:

| Skewness Value | Interpretation |
|--------------|---------------|
| -0.5 to 0.5 | Approximately Normal |
| 0.5 to 1 | Moderately Skewed |
| > 1 | Highly Skewed |
| < -1 | Highly Left-Skewed |

## Key Findings

All analyzed variables exhibit positive skewness, indicating right-skewed distributions.

The most highly skewed variables are:

| Variable | Skewness |
|----------|-----------|
| previous_cancellations | 34.32 |
| babies | 21.15 |
| previous_bookings_not_canceled | 20.46 |
| adults | 19.86 |
| days_in_waiting_list | 19.40 |
| adr | 10.92 |
| booking_changes | 5.55 |

Moderately skewed variables include:

- children (3.46)
- stays_in_week_nights (2.69)
- lead_time (1.43)
- stays_in_weekend_nights (1.41)
- agent (1.12)

## Interpretation

The dataset contains several variables with extremely high positive skewness. This indicates that most observations are concentrated at lower values while a small number of observations have unusually large values.

For example:

- Most guests have few or no previous cancellations, while a small number have many cancellations.
- Most bookings involve a small number of adults and children, while a few bookings involve large groups.
- Most customers make few booking modifications, while a limited number make many changes.

Such patterns are common in hotel booking datasets and often represent genuine customer behavior rather than data quality issues.

## Why Transformation is Considered?

Data transformation is used to reduce skewness and make distributions closer to normal.

Common transformations include:

- Log Transformation
- Square Root Transformation
- Box-Cox Transformation

Benefits:

- Reduces the influence of extreme values.
- Improves distribution symmetry.
- Enhances statistical analysis.
- Can improve machine learning model performance.

## Transformation Decision

Although several variables show high positive skewness, these values represent valid business behavior rather than data entry errors. Since the objective of this project is exploratory data analysis, transformations were not applied at this stage to preserve the original business information contained in the dataset.

## Business Insight

The high skewness observed in booking-related variables indicates that hotel customer behavior is not evenly distributed. A small segment of customers accounts for unusually high booking activity, cancellations, waiting periods, or reservation changes. These exceptional customers may represent important business segments that warrant further analysis.

## Conclusion

Skewness analysis revealed that multiple numerical variables are strongly right-skewed. While transformation techniques could be applied to normalize these distributions, the original values were retained to preserve business interpretability during exploratory data analysis.

In [287]:
df.to_csv("../data/interim/cleaned_day2.csv", index=False)

In [288]:
import os

os.path.exists("../data/interim/cleaned_day2.csv")

True

# Day 2 Summary

## Objective

The objective of Day 2 was to prepare a clean and reliable dataset by correcting data quality issues, handling missing values, detecting outliers, and evaluating the need for data transformation.

## Activities Performed

### 1. Datatype Correction
- Converted `reservation_status_date` from object datatype to datetime format.
- Improved date handling and enabled future time-based analysis.

### 2. Standardization
- Reviewed categorical variables for inconsistent labels.
- Replaced ambiguous categories such as `Undefined` with meaningful business values.

### 3. Duplicate Removal
- Identified and removed duplicate records.
- Reduced redundancy and improved dataset quality.

### 4. Missing Value Treatment
- Dropped the `company` column due to excessive missing values.
- Filled missing values in:
  - `children` using mode.
  - `country` using mode.
  - `agent` using median.
- Verified that no missing values remained in the dataset.

### 5. Outlier Detection
- Applied the IQR method to identify potential outliers.
- Observed significant outliers in variables such as:
  - adults
  - children
  - booking_changes
  - adr
  - lead_time

### 6. Skewness and Transformation Analysis
- Evaluated skewness of numerical variables.
- Identified strong right-skewed distributions in multiple features.
- Determined that transformations were not required at this stage because extreme values represented valid business behavior.

## Final Outcome

The dataset is now:

- Free from duplicate records.
- Free from missing values.
- Datatypes are corrected.
- Categories are standardized.
- Outliers have been identified and documented.
- Ready for exploratory analysis and visualization in subsequent project stages.

## Deliverable

The cleaned dataset was successfully saved as:

`cleaned_day2.csv`

This file will be used as the input dataset for the next stage of the project.

In [289]:
df.duplicated().sum()

np.int64(0)

In [290]:
df[df.duplicated()].head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,booking_changes,deposit_type,agent,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date


In [292]:
df.shape

(87370, 31)

In [293]:
df.duplicated().sum().sum()

np.int64(0)